In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
import matplotlib.pyplot as plt
from pyprojroot import here

In [2]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results"
OUTPUT_DIR = here() / "results" / "validation"

In [3]:
sub_df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_subindex.csv", dtype={"CVEGEO": str})
relms_df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_index.csv", dtype={"CVEGEO": str})

In [4]:
idx_cols = [
    "idx_resources",
    "idx_energy",
    "idx_logistics",
    "idx_market",
    "idx_security",
]

In [5]:
relms_df["CVEGEO"] = relms_df["CVEGEO"].str.zfill(5)
relms_df = relms_df[["CVEGEO", "RELMS_index"]]
sub_df["CVEGEO"] = sub_df["CVEGEO"].str.zfill(5)
sub_df["RELMS_arith"] = sub_df[idx_cols].mean(axis=1)

In [6]:
df = relms_df.merge(sub_df[["CVEGEO", "RELMS_arith"]], on="CVEGEO", how="inner")

In [7]:
df["rank_geo"] = df["RELMS_index"].rank(ascending=False, method="min").astype(int)
df["rank_arith"] = df["RELMS_arith"].rank(ascending=False, method="min").astype(int)
df["rank_shift"] = (df["rank_geo"] - df["rank_arith"]).abs()

In [8]:
rho, p_spearman = spearmanr(df["RELMS_index"], df["RELMS_arith"])
 
print("=" * 60)
print("SPEARMAN RANK CORRELATION (Geometric vs Arithmetic)")
print("=" * 60)
print(f"  rho = {rho:.4f}, p-value = {p_spearman:.4e}")
print()

SPEARMAN RANK CORRELATION (Geometric vs Arithmetic)
  rho = 0.9028, p-value = 0.0000e+00



In [9]:
stat, p_wilcoxon = wilcoxon(df["RELMS_index"], df["RELMS_arith"])
 
print("=" * 60)
print("WILCOXON SIGNED-RANK TEST (Geometric vs Arithmetic scores)")
print("=" * 60)
print(f"  statistic = {stat:.4f}, p-value = {p_wilcoxon:.4e}")
if p_wilcoxon < 0.05:
    print("Significant difference between the two score distributions.")
else:
    print("No significant difference between the two score distributions.")
print()

WILCOXON SIGNED-RANK TEST (Geometric vs Arithmetic scores)
  statistic = 0.0000, p-value = 0.0000e+00
Significant difference between the two score distributions.



In [10]:
n_top = max(1, int(np.ceil(len(df) * 0.05)))
top_divergent = df.sort_values("rank_shift", ascending=False).head(n_top)
 
print("=" * 60)
print(f"TOP {n_top} MUNICIPALITIES (5%) BY RANK SHIFT")
print("=" * 60)
print(
    top_divergent[
        ["CVEGEO", "RELMS_index", "RELMS_arith", "rank_geo", "rank_arith", "rank_shift"]
    ]
    .round(4)
    .to_string(index=False)
)
print()
 
top_divergent.to_csv(OUTPUT_DIR / "top5pct_divergent_municipalities.csv", index=False)

TOP 124 MUNICIPALITIES (5%) BY RANK SHIFT
CVEGEO  RELMS_index  RELMS_arith  rank_geo  rank_arith  rank_shift
 21114       0.3589       0.5482      2264         337        1927
 11020       0.0070       0.5247      2459         654        1805
 15033       0.4234       0.5656      1750         207        1543
 24028       0.3449       0.5120      2318         853        1465
 07119       0.3759       0.5190      2202         740        1462
 26059       0.3518       0.5109      2293         872        1421
 01001       0.3963       0.5264      2040         630        1410
 20490       0.3730       0.5121      2212         850        1362
 09015       0.4074       0.5233      1941         676        1265
 20558       0.4021       0.5182      1989         751        1238
 19003       0.0061       0.4912      2462        1225        1237
 15106       0.4450       0.5598      1475         248        1227
 20109       0.4229       0.5330      1759         535        1224
 20434       0.3861 

In [11]:
fig, ax = plt.subplots(figsize=(7, 7))
 
ax.scatter(
    df["RELMS_arith"],
    df["RELMS_index"],
    s=15,
    alpha=0.4,
    color="steelblue",
    label="All municipalities",
)
ax.scatter(
    top_divergent["RELMS_arith"],
    top_divergent["RELMS_index"],
    s=25,
    color="crimson",
    label=f"Top 5% rank shift (n={n_top})",
)
 
lims = [
    min(df["RELMS_arith"].min(), df["RELMS_index"].min()),
    max(df["RELMS_arith"].max(), df["RELMS_index"].max()),
]
ax.plot(lims, lims, "k--", linewidth=1, label="y = x (perfect agreement)")
 
ax.set_xlabel("Arithmetic Mean Composite (RELMS_arith)")
ax.set_ylabel("Geometric Mean Composite (RELMS_index)")
#ax.set_title("Geometric vs Arithmetic Mean Aggregation\n(RELMS with safety index)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "geometric_vs_arithmetic_scatter.png", dpi=300)
plt.close(fig)
 
print(f"Plots and top-divergent CSV saved to: {OUTPUT_DIR.resolve()}")

Plots and top-divergent CSV saved to: C:\Users\franc\OneDrive\Documents\Repositories\hydrogen-hub-index\results\validation
